# Введение в MapReduce модель на Python


In [ ]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [3]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [4]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [5]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [6]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [7]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [8]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [9]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [10]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [11]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [12]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [13]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [14]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [15]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [16]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.098046006120433)),
 (1, np.float64(2.098046006120433)),
 (2, np.float64(2.098046006120433)),
 (3, np.float64(2.098046006120433)),
 (4, np.float64(2.098046006120433))]

## Inverted index

In [17]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [18]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [19]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('a', 2), ('is', 18), ('it', 18), ('what', 10)]),
 (1, [('banana', 2)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.007361861584142537)),
   (None, np.float64(0.04138949368239209)),
   (None, np.float64(0.043085495008126085)),
   (None, np.float64(0.04836264458488082)),
   (None, np.float64(0.11435461414941783)),
   (None, np.float64(0.12413378751563475)),
   (None, np.float64(0.1536376183910627)),
   (None, np.float64(0.22469363109880047)),
   (None, np.float64(0.3252467054049234)),
   (None, np.float64(0.3572935877486194)),
   (None, np.float64(0.39131068626120147)),
   (None, np.float64(0.4069601531607582)),
   (None, np.float64(0.43470104238625407)),
   (None, np.float64(0.4796207053185667))]),
 (1,
  [(None, np.float64(0.5127441636759409)),
   (None, np.float64(0.5659830900258485)),
   (None, np.float64(0.5676951107696212)),
   (None, np.float64(0.6363115037742982)),
   (None, np.float64(0.6368102894682065)),
   (None, np.float64(0.6675712105728077)),
   (None, np.float64(0.6852283686916533)),
   (None, np.float64(0.6905732096880716)),
   (None, np.float64(0.71837799

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [ ]:
def RECORDREADER(data):
    return data

def MAP(key, value):
    return ('max', value)

def REDUCE(key, values):
    return max(values)

def MapReduce(data):
    mapped = []
    for key, value in enumerate(RECORDREADER(data)):
        mapped.append(MAP(key, value))
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    result = {}
    for k, v in grouped.items():
        result[k] = REDUCE(k, v)
    
    return result

data = [3, 7, 2, 9, 1, 5]
print(MapReduce(data))

{'max': 9}


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [21]:
def RECORDREADER(data):
    return data

def MAP(key, value):
    return ('sum', (value, 1))

def REDUCE(key, values):
    total_sum = sum(v[0] for v in values)
    total_count = sum(v[1] for v in values)
    return total_sum / total_count

def MapReduce(data):
    mapped = []
    for key, value in enumerate(RECORDREADER(data)):
        mapped.append(MAP(key, value))
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    result = {}
    for k, v in grouped.items():
        result[k] = REDUCE(k, v)
    
    return result

data = [3, 7, 2, 9, 1, 5]
print(MapReduce(data))

{'sum': 4.5}


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [22]:
def RECORDREADER(data):
    return data

def MAP(key, value):
    return (key, value)

def REDUCE(key, values):
    return list(values)

def MapReduce(data):
    mapped = []
    for key, value in enumerate(RECORDREADER(data)):
        mapped.append(MAP(key, value))
    
    mapped.sort(key=lambda x: x[0])
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    result = {}
    for k, v in grouped.items():
        result[k] = REDUCE(k, v)
    
    return result

data = [('a', 1), ('b', 2), ('a', 3), ('c', 4), ('b', 5)]
print(MapReduce(data))

{0: [('a', 1)], 1: [('b', 2)], 2: [('a', 3)], 3: [('c', 4)], 4: [('b', 5)]}


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [23]:
def RECORDREADER(data):
    return data

def MAP(key, value):
    return (value, None)

def REDUCE(key, values):
    return key

def MapReduce(data):
    mapped = []
    for key, value in enumerate(RECORDREADER(data)):
        mapped.append(MAP(key, value))
    
    mapped.sort(key=lambda x: x[0])
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    result = []
    for k, v in grouped.items():
        result.append(REDUCE(k, v))
    
    return result

data = [1, 2, 3, 2, 1, 4, 3, 5]
print(MapReduce(data))

[1, 2, 3, 4, 5]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [24]:
def RECORDREADER(data):
    return data

def MAP(key, value, predicate):
    if predicate(value):
        return (value, value)
    return None

def REDUCE(key, values):
    return values[0]

def MapReduce(data, predicate):
    mapped = []
    for key, value in enumerate(RECORDREADER(data)):
        result = MAP(key, value, predicate)
        if result is not None:
            mapped.append(result)
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    result = {}
    for k, v in grouped.items():
        result[k] = REDUCE(k, v)
    
    return list(result.values())

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [25]:
def RECORDREADER(data):
    return data

def MAP(key, value, attributes):
    projected = tuple(value[i] for i in attributes)
    return (projected, projected)

def REDUCE(key, values):
    return values[0]

def MapReduce(data, attributes):
    mapped = []
    for key, value in enumerate(RECORDREADER(data)):
        result = MAP(key, value, attributes)
        mapped.append(result)
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    result = {}
    for k, v in grouped.items():
        result[k] = REDUCE(k, v)
    
    return list(result.values())

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [26]:
def RECORDREADER(data):
    return data

def MAP(key, value):
    return (value, value)

def REDUCE(key, values):
    return (key, key)

def MapReduce(data1, data2):
    mapped = []
    for key, value in enumerate(RECORDREADER(data1)):
        mapped.append(MAP(key, value))
    for key, value in enumerate(RECORDREADER(data2)):
        mapped.append(MAP(key, value))
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    result = {}
    for k, v in grouped.items():
        result[k] = REDUCE(k, v)
    
    return list(result.keys())

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [ ]:
def RECORDREADER(data):
    return data

def MAP(key, value):
    return (value, value)

def REDUCE(key, values):
    if len(values) == 2:
        return (key, key)
    return None

def MapReduce(data1, data2):
    mapped = []
    for key, value in enumerate(RECORDREADER(data1)):
        mapped.append(MAP(key, value))
    for key, value in enumerate(RECORDREADER(data2)):
        mapped.append(MAP(key, value))
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    result = []
    for k, v in grouped.items():
        reduced = REDUCE(k, v)
        if reduced is not None:
            result.append(k)
    
    return result

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [27]:
def RECORDREADER(data):
    return data

def MAP(key, value, source):
    return (value, source)

def REDUCE(key, values):
    if values == ['R']:
        return (key, key)
    return None

def MapReduce(dataR, dataS):
    mapped = []
    for key, value in enumerate(RECORDREADER(dataR)):
        mapped.append(MAP(key, value, 'R'))
    for key, value in enumerate(RECORDREADER(dataS)):
        mapped.append(MAP(key, value, 'S'))
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    result = []
    for k, v in grouped.items():
        reduced = REDUCE(k, v)
        if reduced is not None:
            result.append(k)
    
    return result

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [28]:
def RECORDREADER(data):
    return data

def MAP(key, value, relation):
    if relation == 'R':
        b, a = value
        return (b, ('R', a))
    else:
        b, c = value
        return (b, ('S', c))

def REDUCE(key, values):
    r_values = [v[1] for v in values if v[0] == 'R']
    s_values = [v[1] for v in values if v[0] == 'S']
    result = []
    for a in r_values:
        for c in s_values:
            result.append((a, key, c))
    return result

def MapReduce(dataR, dataS):
    mapped = []
    for key, value in enumerate(RECORDREADER(dataR)):
        mapped.append(MAP(key, value, 'R'))
    for key, value in enumerate(RECORDREADER(dataS)):
        mapped.append(MAP(key, value, 'S'))
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    result = []
    for k, v in grouped.items():
        result.extend(REDUCE(k, v))
    
    return result

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [29]:
def RECORDREADER(data):
    return data

def MAP(key, value):
    a, b, c = value
    return (a, b)

def REDUCE(key, values, operation):
    if operation == 'SUM':
        x = sum(values)
    elif operation == 'MAX':
        x = max(values)
    elif operation == 'MIN':
        x = min(values)
    elif operation == 'COUNT':
        x = len(values)
    elif operation == 'AVG':
        x = sum(values) / len(values)
    else:
        x = values
    return (key, x)

def MapReduce(data, operation):
    mapped = []
    for key, value in enumerate(RECORDREADER(data)):
        mapped.append(MAP(key, value))
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    result = {}
    for k, v in grouped.items():
        result[k] = REDUCE(k, v, operation)
    
    return result

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [30]:
def RECORDREADER(data):
    return data

def MAP(key, value):
    if value[0] == 'M':
        _, i, j, mij = value
        return (j, ('M', i, mij))
    else:
        _, j, vj = value
        return (j, ('V', vj))

def REDUCE(key, values):
    m_values = [(v[1], v[2]) for v in values if v[0] == 'M']
    v_values = [v[1] for v in values if v[0] == 'V']
    
    result = []
    if v_values:
        v = v_values[0]
        for i, mij in m_values:
            result.append((i, mij * v))
    return result

def MapReduce(matrix_data, vector_data):
    mapped = []
    for key, value in enumerate(RECORDREADER(matrix_data)):
        mapped.append(MAP(key, value))
    for key, value in enumerate(RECORDREADER(vector_data)):
        mapped.append(MAP(key, value))
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    partial_result = []
    for k, v in grouped.items():
        partial_result.extend(REDUCE(k, v))
    
    final = {}
    for i, val in partial_result:
        if i not in final:
            final[i] = 0
        final[i] += val
    
    return final

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [31]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [33]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J)
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
    (j, k) = k1
    w = v1
    for i in range(small_mat.shape[0]):
        yield ((i, k), small_mat[i, j] * w)

def REDUCE(key, values):
    (i, k) = key
    yield ((i, k), sum(values))

Проверьте своё решение

In [34]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [35]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [36]:
def RECORDREADER(matrices):
    return matrices

def MAP(key, value):
    matrix_id, i, j, val = value
    if matrix_id == 'A':
        return (j, ('A', i, val))
    else:
        return (j, ('B', i, val))

def REDUCE(key, values):
    a_elements = [(v[1], v[2]) for v in values if v[0] == 'A']
    b_elements = [(v[1], v[2]) for v in values if v[0] == 'B']
    
    result = []
    for i, a_val in a_elements:
        for k, b_val in b_elements:
            result.append(((i, k), a_val * b_val))
    return result

def MapReduce(matrixA, matrixB):
    data = []
    for i, row in enumerate(matrixA):
        for j, val in enumerate(row):
            data.append(('A', i, j, val))
    for i, row in enumerate(matrixB):
        for j, val in enumerate(row):
            data.append(('B', i, j, val))
    
    mapped = []
    for key, value in enumerate(RECORDREADER(data)):
        mapped.append(MAP(key, value))
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    partial_results = []
    for k, v in grouped.items():
        partial_results.extend(REDUCE(k, v))
    
    final = {}
    for (i, k), val in partial_results:
        if (i, k) not in final:
            final[(i, k)] = 0
        final[(i, k)] += val
    
    return final

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [37]:
def RECORDREADER_A(matrixA):
    for i, row in enumerate(matrixA):
        for j, val in enumerate(row):
            yield ('A', i, j, val)

def RECORDREADER_B(matrixB):
    for i, row in enumerate(matrixB):
        for j, val in enumerate(row):
            yield ('B', i, j, val)

def MAP(key, value):
    matrix_id, i, j, val = value
    if matrix_id == 'A':
        return (j, ('A', i, val))
    else:
        return (i, ('B', j, val))

def REDUCE(key, values):
    a_elements = [(v[1], v[2]) for v in values if v[0] == 'A']
    b_elements = [(v[1], v[2]) for v in values if v[0] == 'B']
    
    result = []
    for i, a_val in a_elements:
        for k, b_val in b_elements:
            result.append(((i, k), a_val * b_val))
    return result

def MapReduce(matrixA, matrixB):
    mapped = []
    
    for key, value in enumerate(RECORDREADER_A(matrixA)):
        mapped.append(MAP(key, value))
    
    for key, value in enumerate(RECORDREADER_B(matrixB)):
        mapped.append(MAP(key, value))
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    partial_results = []
    for k, v in grouped.items():
        partial_results.extend(REDUCE(k, v))
    
    final = {}
    for (i, k), val in partial_results:
        if (i, k) not in final:
            final[(i, k)] = 0
        final[(i, k)] += val
    
    return final

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [38]:
import random

def RECORDREADER_A1(matrixA):
    for i, row in enumerate(matrixA):
        for j, val in enumerate(row):
            if random.random() > 0.5:
                yield ('A', i, j, val)

def RECORDREADER_A2(matrixA):
    for i, row in enumerate(matrixA):
        for j, val in enumerate(row):
            if random.random() <= 0.5:
                yield ('A', i, j, val)

def RECORDREADER_B1(matrixB):
    for i, row in enumerate(matrixB):
        for j, val in enumerate(row):
            if random.random() > 0.5:
                yield ('B', i, j, val)

def RECORDREADER_B2(matrixB):
    for i, row in enumerate(matrixB):
        for j, val in enumerate(row):
            if random.random() <= 0.5:
                yield ('B', i, j, val)

def MAP(key, value):
    matrix_id, i, j, val = value
    if matrix_id == 'A':
        return (j, ('A', i, val))
    else:
        return (i, ('B', j, val))

def REDUCE(key, values):
    a_elements = [(v[1], v[2]) for v in values if v[0] == 'A']
    b_elements = [(v[1], v[2]) for v in values if v[0] == 'B']
    
    result = []
    for i, a_val in a_elements:
        for k, b_val in b_elements:
            result.append(((i, k), a_val * b_val))
    return result

def MapReduce(matrixA, matrixB):
    mapped = []
    
    recordreaders_a = [RECORDREADER_A1(matrixA), RECORDREADER_A2(matrixA)]
    recordreaders_b = [RECORDREADER_B1(matrixB), RECORDREADER_B2(matrixB)]
    
    key = 0
    for reader in recordreaders_a:
        for value in reader:
            mapped.append(MAP(key, value))
            key += 1
    
    for reader in recordreaders_b:
        for value in reader:
            mapped.append(MAP(key, value))
            key += 1
    
    grouped = {}
    for k, v in mapped:
        if k not in grouped:
            grouped[k] = []
        grouped[k].append(v)
    
    partial_results = []
    for k, v in grouped.items():
        partial_results.extend(REDUCE(k, v))
    
    final = {}
    for (i, k), val in partial_results:
        if (i, k) not in final:
            final[(i, k)] = 0
        final[(i, k)] += val
    
    return final

matrixA = [[1, 2], [3, 4]]
matrixB = [[5, 6], [7, 8]]

result = MapReduce(matrixA, matrixB)
print(result)

{(0, 0): 38, (0, 1): 76, (1, 0): 58, (1, 1): 100}
